# NC Amenities and Accessibility

This notebook prepares amenity and recreation data for our North Carolina
housing recommendation system. We will inspect and clean OpenStreetMap
and PAD-US data, retrieve transportation and coastal-access data, and
calculate accessibility features for housing areas.

In [1]:
%pip install geopandas pyogrio requests

Note: you may need to restart the kernel to use updated packages.


In [2]:
from pathlib import Path

import pandas as pd
import geopandas as gpd
import pyogrio
import requests

# Assumes this notebook is running from the project folder
DATA_DIR = Path("data")

OSM_PATH = DATA_DIR / "north-carolina.gpkg"
PADUS_PATH = DATA_DIR / "PADUS4_1_StateNC.gdb"

for path in [OSM_PATH, PADUS_PATH]:
    print(f"{path}: {'Found' if path.exists() else 'NOT FOUND'}")

print("\nWorking directory:", Path.cwd())

data\north-carolina.gpkg: Found
data\PADUS4_1_StateNC.gdb: Found

Working directory: c:\Users\kyleh\OneDrive\Desktop\final_group_project\2302-group-proj


In [3]:
# Lists layers without loading all their geographic features
osm_layers = pd.DataFrame(
    pyogrio.list_layers(OSM_PATH),
    columns=["layer", "geometry_type"]
)

padus_layers = pd.DataFrame(
    pyogrio.list_layers(PADUS_PATH),
    columns=["layer", "geometry_type"]
)

print("OpenStreetMap layers")
display(osm_layers)

print("PAD-US layers")
display(padus_layers)

OpenStreetMap layers


,layer,geometry_type
0,gis_osm_traffic_free,Point
1,gis_osm_places_free,Point
2,gis_osm_natural_free,Point
3,gis_osm_pois_free,Point
4,gis_osm_transport_free,Point
5,gis_osm_pofw_free,Point
6,gis_osm_roads_free,LineString
7,gis_osm_railways_free,LineString
8,gis_osm_waterways_free,LineString
9,gis_osm_water_a_free,MultiPolygon


PAD-US layers


,layer,geometry_type
0,PADUS4_1Fee_State_NC,MultiPolygon
1,PADUS4_1Easement_State_NC,MultiPolygon
2,PADUS4_1Proclamation_State_NC,MultiPolygon
3,PADUS4_1Comb_DOD_Trib_NGP_Fee_Desig_Ease_State_NC,MultiPolygon
4,PADUS4_1Designation_State_NC,MultiPolygon
5,PADUS4_1Marine,MultiPolygon
6,Public_Access,NaN
7,Agency_Name,NaN
8,Agency_Type,NaN
9,Category,NaN


In [4]:
# Businesses and amenities can be mapped as points or area outlines
osm_poi_points = gpd.read_file(
    OSM_PATH,
    layer="gis_osm_pois_free",
    engine="pyogrio"
)

osm_poi_areas = gpd.read_file(
    OSM_PATH,
    layer="gis_osm_pois_a_free",
    engine="pyogrio"
)

for label, df in [
    ("POI points", osm_poi_points),
    ("POI areas", osm_poi_areas)
]:
    print(f"\n{label}: {len(df):,} records")
    print("CRS:", df.crs)
    print("Columns:", df.columns.tolist())
    display(df.head(3))


POI points: 54,460 records
CRS: GEOGCS["WGS 84 (CRS84)",DATUM["WGS_1984",SPHEROID["WGS 84",6378137,298.257223563]],PRIMEM["Greenwich",0],UNIT["degree",0.0174532925199433,AUTHORITY["EPSG","9122"]],AXIS["Longitude",EAST],AXIS["Latitude",NORTH],AUTHORITY["OGC","CRS84"]]
Columns: ['osm_id', 'code', 'fclass', 'name', 'geometry']


,osm_id,code,fclass,name,geometry
0,153913728,2204,park,Brooklawn Park,POINT (-82.54117 35.52071)
1,154401578,2204,park,Rosebank Park,POINT (-82.53948 35.53708)
2,169137017,2742,viewpoint,Air Bellows Overlook,POINT (-81.12275 36.43045)



POI areas: 67,530 records
CRS: GEOGCS["WGS 84 (CRS84)",DATUM["WGS_1984",SPHEROID["WGS 84",6378137,298.257223563]],PRIMEM["Greenwich",0],UNIT["degree",0.0174532925199433,AUTHORITY["EPSG","9122"]],AXIS["Longitude",EAST],AXIS["Latitude",NORTH],AUTHORITY["OGC","CRS84"]]
Columns: ['osm_id', 'code', 'fclass', 'name', 'geometry']


,osm_id,code,fclass,name,geometry
0,16428521,2015,graveyard,Olive Hill Cemetery,"MULTIPOLYGON (((-81.69119 35.73627, -81.68966 ..."
1,16520053,2258,track,,"MULTIPOLYGON (((-78.00081 34.77249, -78.00077 ..."
2,16641438,2252,pitch,,"MULTIPOLYGON (((-79.18999 35.53488, -79.18988 ..."


In [5]:
category_counts = pd.concat(
    [
        osm_poi_points["fclass"].value_counts().rename("points"),
        osm_poi_areas["fclass"].value_counts().rename("areas")
    ],
    axis=1
).fillna(0).astype(int)

category_counts["total_records"] = category_counts.sum(axis=1)

# These are raw record counts, not yet deduplicated venue counts
display(
    category_counts.sort_values(
        "total_records", ascending=False
    ).rename_axis("category")
)

,points,areas,total_records
category,,,
pitch,432,17927,18359
restaurant,5144,2671,7815
graveyard,36,7065,7101
bench,5923,9,5932
swimming_pool,808,4958,5766
...,...,...,...
consulate,3,1,4
castle,1,2,3
recycling_metal,1,2,3


In [6]:
PADUS_LAYER = "PADUS4_1Comb_DOD_Trib_NGP_Fee_Desig_Ease_State_NC"

# Read a small sample first
padus_sample = gpd.read_file(
    PADUS_PATH,
    layer=PADUS_LAYER,
    rows=5,
    engine="pyogrio"
)

print("PAD-US columns:")
print(padus_sample.columns.tolist())
display(padus_sample)

# Read the lookup table that explains public-access codes
public_access_lookup = pyogrio.read_dataframe(
    PADUS_PATH,
    layer="Public_Access",
    read_geometry=False
)

print("\nPublic-access definitions:")
display(public_access_lookup)

PAD-US columns:
['FeatClass', 'Category', 'Own_Type', 'Own_Name', 'Loc_Own', 'Mang_Type', 'Mang_Name', 'Loc_Mang', 'Des_Tp', 'Loc_Ds', 'Unit_Nm', 'Loc_Nm', 'State_Nm', 'Agg_Src', 'GIS_Src', 'Src_Date', 'GIS_Acres', 'Source_PAID', 'WDPA_Cd', 'Pub_Access', 'Access_Src', 'Access_Dt', 'GAP_Sts', 'GAPCdSrc', 'GAPCdDt', 'IUCN_Cat', 'IUCNCtSrc', 'IUCNCtDt', 'Date_Est', 'Comments', 'Term', 'Duration', 'd_FeatClass', 'd_Category', 'd_Own_Type', 'd_Own_Name', 'd_Mang_Type', 'd_Mang_Name', 'd_Des_Tp', 'd_State_Nm', 'd_Pub_Access', 'd_GAP_Sts', 'd_IUCN_Cat', 'Shape_Length', 'Shape_Area', 'geometry']


,FeatClass,Category,Own_Type,Own_Name,Loc_Own,Mang_Type,Mang_Name,Loc_Mang,Des_Tp,Loc_Ds,...,d_Mang_Type,d_Mang_Name,d_Des_Tp,d_State_Nm,d_Pub_Access,d_GAP_Sts,d_IUCN_Cat,Shape_Length,Shape_Area,geometry
0,Proclamation,Proclamation,DESG,DESG,UNK,FED,DOD,,MIL,,...,Federal,Department of Defense,Military Land,North Carolina,Closed,4 - no known mandate for biodiversity protection,Unassigned,13255.792227,3.655659e+06,"MULTIPOLYGON (((1713093.022 1464974.124, 17130..."
1,Proclamation,Proclamation,DESG,DESG,UNK,FED,DOD,,MIL,,...,Federal,Department of Defense,Military Land,North Carolina,Closed,4 - no known mandate for biodiversity protection,Unassigned,6.281512,3.137474e+00,"MULTIPOLYGON (((1519694.406 1575162.508, 15196..."
2,Proclamation,Proclamation,DESG,DESG,UNK,FED,DOD,,MIL,,...,Federal,Department of Defense,Military Land,North Carolina,Closed,4 - no known mandate for biodiversity protection,Unassigned,38905.484753,3.225931e+07,"MULTIPOLYGON (((1485215.256 1462497.526, 14852..."
3,Proclamation,Proclamation,DESG,DESG,UNK,FED,DOD,,MIL,,...,Federal,Department of Defense,Military Land,North Carolina,Closed,4 - no known mandate for biodiversity protection,Unassigned,3214.141157,4.267166e+05,"MULTIPOLYGON (((1354715.685 1459156.11, 135472..."
4,Proclamation,Proclamation,DESG,DESG,UNK,FED,DOD,,MIL,,...,Federal,Department of Defense,Military Land,North Carolina,Closed,4 - no known mandate for biodiversity protection,Unassigned,63079.678638,1.886887e+08,"MULTIPOLYGON (((1794134.22 1602904.488, 179413..."



Public-access definitions:


,Code,Dom
0,OA,Open Access
1,RA,Restricted Access
2,XA,Closed
3,UK,Unknown


In [7]:
# Print all categories without pandas hiding the middle rows
print(
    category_counts
    .sort_values("total_records", ascending=False)
    .to_string()
)

                     points  areas  total_records
fclass                                           
pitch                   432  17927          18359
restaurant             5144   2671           7815
graveyard                36   7065           7101
bench                  5923      9           5932
swimming_pool           808   4958           5766
fast_food              3146   2180           5326
playground              577   3966           4543
camera_surveillance    4512      1           4513
park                    380   3410           3790
school                 1583   1720           3303
convenience            1545   1392           2937
shelter                 336   2528           2864
waste_basket           2609      0           2609
bank                    676   1085           1761
hotel                   147   1545           1692
toilet                  784    886           1670
tourist_info           1553    117           1670
supermarket             904    747           1651


In [8]:
padus = gpd.read_file(
    PADUS_PATH,
    layer=PADUS_LAYER,
    engine="pyogrio"
)

print(f"PAD-US records: {len(padus):,}")
print("CRS:", padus.crs)

display(
    padus.groupby(
        ["Pub_Access", "d_Pub_Access"],
        dropna=False
    ).size().rename("records").reset_index()
)

c:\Users\kyleh\anaconda3\Lib\site-packages\pyogrio\raw.py:200: RuntimeWarning: organizePolygons() received a polygon with more than 100 parts.  The processing may be really slow.  You can skip the processing by setting METHOD=SKIP.
  return ogr_read(


PAD-US records: 14,576
CRS: PROJCS["USA_Contiguous_Albers_Equal_Area_Conic_USGS_version",GEOGCS["NAD83",DATUM["North_American_Datum_1983",SPHEROID["GRS 1980",6378137,298.257222101,AUTHORITY["EPSG","7019"]],AUTHORITY["EPSG","6269"]],PRIMEM["Greenwich",0,AUTHORITY["EPSG","8901"]],UNIT["degree",0.0174532925199433,AUTHORITY["EPSG","9122"]],AUTHORITY["EPSG","4269"]],PROJECTION["Albers_Conic_Equal_Area"],PARAMETER["latitude_of_center",23],PARAMETER["longitude_of_center",-96],PARAMETER["standard_parallel_1",29.5],PARAMETER["standard_parallel_2",45.5],PARAMETER["false_easting",0],PARAMETER["false_northing",0],UNIT["metre",1,AUTHORITY["EPSG","9001"]],AXIS["Easting",EAST],AXIS["Northing",NORTH],AUTHORITY["ESRI","102039"]]


,Pub_Access,d_Pub_Access,records
0,OA,Open Access,2830
1,RA,Restricted Access,1037
2,UK,Unknown,6611
3,XA,Closed,4098


In [9]:
# Keep restricted-access records separately for possible later use
padus_open = padus[
    padus["Pub_Access"].eq("OA")
].copy()

padus_restricted = padus[
    padus["Pub_Access"].eq("RA")
].copy()

# Remove missing or empty geometries
padus_open = padus_open[
    padus_open.geometry.notna()
    & ~padus_open.geometry.is_empty
].copy()

# Project to NC State Plane in meters for later distance calculations
padus_open = padus_open.to_crs("EPSG:32119")

print(f"Open-access land records: {len(padus_open):,}")

display(
    padus_open["d_Des_Tp"]
    .value_counts(dropna=False)
    .rename_axis("designation")
    .reset_index(name="records")
)

display(
    padus_open[
        ["Unit_Nm", "d_Des_Tp", "d_Pub_Access"]
    ].head(10)
)

Open-access land records: 2,830


,designation,records
0,Local Park,1290
1,Local Recreation Area,937
2,Local Conservation Area,177
3,State Conservation Area,69
4,State Other or Unknown,69
5,Local Other or Unknown,65
6,State Park,62
7,Inventoried Roadless Area,44
8,Local Resource Management Area,19
9,State Resource Management Area,12


,Unit_Nm,d_Des_Tp,d_Pub_Access
23,Cherokee National Forest,National Forest,Open Access
24,Chattahoochee National Forest,National Forest,Open Access
25,Cherokee National Forest,National Forest,Open Access
26,Croatan National Forest,National Forest,Open Access
27,Nantahala National Forest,National Forest,Open Access
28,Pisgah National Forest,National Forest,Open Access
29,Sumter National Forest,National Forest,Open Access
30,Uwharrie National Forest,National Forest,Open Access
31,Nantahala Purchase Unit,National Forest,Open Access
38,Cape Hatteras National Seashore,National Lakeshore or Seashore,Open Access


In [10]:
# Project-defined groups for the recommender
land_groups = {
    "parks_and_recreation": [
        "Local Park",
        "Local Recreation Area",
        "State Park",
        "State Recreation Area",
        "Recreation Management Area",
        "Access Area"
    ],
    "forests_and_nature": [
        "National Forest",
        "National Park",
        "Inventoried Roadless Area",
        "Local Conservation Area",
        "State Conservation Area",
        "Conservation Area",
        "Private Conservation",
        "National Public Lands",
        "Local Resource Management Area",
        "State Resource Management Area",
        "Watershed Protection Area"
    ],
    "coastal_and_river_lands": [
        "National Lakeshore or Seashore",
        "Wild and Scenic River"
    ],
    "historic_and_cultural": [
        "Local Historic or Cultural Area",
        "State Historic or Cultural Area"
    ]
}

designation_to_group = {
    designation: group
    for group, designations in land_groups.items()
    for designation in designations
}

padus_open["amenity_group"] = (
    padus_open["d_Des_Tp"]
    .map(designation_to_group)
    .fillna("other_open_land")
)

display(
    padus_open.groupby("amenity_group")
    .size()
    .reset_index(name="land_records")
)

,amenity_group,land_records
0,coastal_and_river_lands,10
1,forests_and_nature,344
2,historic_and_cultural,12
3,other_open_land,163
4,parks_and_recreation,2301


In [11]:
print(category_counts.sort_values("total_records", ascending=False).to_string())

                     points  areas  total_records
fclass                                           
pitch                   432  17927          18359
restaurant             5144   2671           7815
graveyard                36   7065           7101
bench                  5923      9           5932
swimming_pool           808   4958           5766
fast_food              3146   2180           5326
playground              577   3966           4543
camera_surveillance    4512      1           4513
park                    380   3410           3790
school                 1583   1720           3303
convenience            1545   1392           2937
shelter                 336   2528           2864
waste_basket           2609      0           2609
bank                    676   1085           1761
hotel                   147   1545           1692
toilet                  784    886           1670
tourist_info           1553    117           1670
supermarket             904    747           1651


In [12]:
# Candidate categories to check against your actual OSM data
candidate_groups = {
    "food_and_nightlife": [
        "restaurant", "fast_food", "cafe", "pub", "bar",
        "biergarten", "nightclub", "food_court"
    ],
    "shopping_and_daily_needs": [
        "supermarket", "convenience", "pharmacy",
        "mall", "department_store", "marketplace"
    ],
    "sports_and_fitness": [
        "sports_centre", "stadium", "pitch", "track",
        "golf_course", "swimming_pool", "fitness_centre"
    ],
    "entertainment_and_culture": [
        "cinema", "theatre", "museum", "arts_centre",
        "theme_park", "zoo", "attraction"
    ],
    "outdoor_recreation": [
        "park", "playground", "picnic_site",
        "viewpoint", "camp_site"
    ]
}

# Only display categories that actually exist in your file
for group, categories in candidate_groups.items():
    available = category_counts.index.intersection(categories)

    print(f"\n{group}")
    print(category_counts.loc[available].to_string())


food_and_nightlife
            points  areas  total_records
fclass                                  
restaurant    5144   2671           7815
fast_food     3146   2180           5326
cafe           921    285           1206
bar            463    232            695
pub            171     93            264
nightclub       32     24             56
food_court       9     20             29
biergarten       6      8             14

shopping_and_daily_needs
                  points  areas  total_records
fclass                                        
convenience         1545   1392           2937
supermarket          904    747           1651
pharmacy             411    177            588
department_store     243    124            367
marketplace           61     55            116
mall                   8    149            157

sports_and_fitness
                points  areas  total_records
fclass                                      
swimming_pool      808   4958           5766
fitness_centr

In [13]:
# Keep the source layer so we can trace each record
points = osm_poi_points.copy()
points["source_layer"] = "gis_osm_pois_free"

areas = osm_poi_areas.copy()
areas["source_layer"] = "gis_osm_pois_a_free"

# Use the same projected CRS for future distance calculations
points = points.to_crs("EPSG:32119")
areas = areas.to_crs("EPSG:32119")

osm_amenities = gpd.GeoDataFrame(
    pd.concat([points, areas], ignore_index=True),
    geometry="geometry",
    crs="EPSG:32119"
)

# Assign our project categories
category_to_group = {
    category: group
    for group, categories in candidate_groups.items()
    for category in categories
}

osm_amenities["amenity_group"] = (
    osm_amenities["fclass"].map(category_to_group)
)

# Keep selected categories and usable geometry
osm_amenities = osm_amenities[
    osm_amenities["amenity_group"].notna()
    & osm_amenities.geometry.notna()
    & ~osm_amenities.geometry.is_empty
].copy()

# Clean names without dropping unnamed amenities
osm_amenities["name"] = (
    osm_amenities["name"]
    .astype("string")
    .str.strip()
    .replace("", pd.NA)
)

# Traceable record ID, not yet a unique venue ID
osm_amenities["record_id"] = (
    osm_amenities["source_layer"]
    + ":"
    + osm_amenities["osm_id"].astype("string")
)

print(f"Selected amenity records: {len(osm_amenities):,}")

display(
    osm_amenities.groupby("amenity_group")
    .size()
    .reset_index(name="records")
)

Selected amenity records: 60,376


,amenity_group,records
0,entertainment_and_culture,1385
1,food_and_nightlife,15405
2,outdoor_recreation,10110
3,shopping_and_daily_needs,5816
4,sports_and_fitness,27660



## Progress Summary: Preparing NC Amenities and Recreation Data

The purpose of this notebook is to prepare location-based features for our North Carolina housing recommendation system. Housing costs help us identify areas within a person’s budget, but the surrounding activities and services also influence where someone wants to live. This notebook focuses on those preferences, including access to food, shopping, sports, entertainment, parks, and other outdoor destinations.

### Loading and inspecting the source data

We began by setting up the Python packages used to work with tabular and geographic data: pandas, GeoPandas, pyogrio, and requests. We defined relative paths to the data folder so the notebook can run on different team members’ computers without relying on one person’s full file path.

We checked that two source datasets were available:

- The North Carolina OpenStreetMap GeoPackage, which contains mapped businesses, amenities, transportation features, and geographic features.
- The North Carolina Protected Areas Database of the United States, or PAD-US, which contains protected land boundaries, ownership and management information, land designations, and public-access classifications.

Before loading large amounts of data, we listed the layers in each file. OpenStreetMap separates features into layers such as points of interest, transportation, roads, waterways, and protected areas. PAD-US contains geographic layers for different land categories, along with lookup tables that explain its classification codes.

### Inspecting OpenStreetMap amenities

We loaded the point and polygon versions of OpenStreetMap’s points-of-interest layer. Both are needed because a business or amenity may be mapped as a single location or as an outline of its building or property.

The point layer contained 54,460 records, and the polygon layer contained 67,530 records. Both included an OSM identifier, a feature code, a category stored in `fclass`, a name, and geographic geometry.

We counted records by category to understand which amenities were represented. Examples included restaurants, cafés, supermarkets, pharmacies, parks, playgrounds, sports pitches, and swimming pools. These initial counts describe mapped records, not necessarily unique businesses or publicly accessible facilities.

We then defined five starting groups for the recommender:

- Food and nightlife
- Shopping and daily needs
- Sports and fitness
- Entertainment and culture
- Outdoor recreation

These are project-defined groups that organize more specific OSM categories. We retained the original `fclass` values so later recommendations can distinguish preferences such as grocery access, cafés, or nightlife rather than treating every amenity in a broad group as interchangeable.

### Inspecting PAD-US public access

We inspected a sample of the combined PAD-US land layer and read its public-access lookup table. The access codes distinguish open access (`OA`), restricted access (`RA`), closed land (`XA`), and unknown access (`UK`).

We separated open-access records from restricted-access records. After removing missing or empty geometries, the open-access subset contained 2,830 land records. Its designations included local parks, local recreation areas, state parks, national forests, conservation areas, and national seashores.

We also defined broader land groups for the recommender, including parks and recreation, forests and nature, coastal and river lands, and historic and cultural areas. Other designations remain labeled as other open land rather than being forced into an unsuitable category.

### Preparing data for accessibility analysis

The OSM preparation step combines selected point and polygon amenities into one GeoDataFrame and assigns their recommendation groups. It also cleans blank names while retaining unnamed features, preserves the source layer, and constructs traceable record identifiers.

We use EPSG:32119, an NC State Plane coordinate system measured in meters, to prepare the geographic data for distance calculations. Original point and polygon geometries are retained at this stage.

### Limitations and remaining work

The data is still being prepared. We have not yet calculated accessibility measures, merged amenities with housing areas, or produced recommendations.

Several checks remain necessary. An OSM point and polygon may describe the same venue, so duplicate representations could inflate nearby counts. PAD-US can contain multiple land pieces and overlapping designations, meaning its records cannot be counted directly as unique parks. We should not remove records solely because they share a name.

Public access also requires care. The OSM fields loaded here do not establish whether a pool, sports field, or other facility is open to the public. PAD-US open-access status does not confirm particular activities, entrances, or current operating conditions.

Next, we will check duplicate records and geographic coverage, retrieve coastal-access and transportation data, and calculate clearly labeled distance and amenity-count features. Those results will later connect to the housing data and support recommendations based on each user’s preferences.

In [14]:
# Exact repeated records within the same source layer
exact_duplicates = osm_amenities.duplicated(
    subset=["source_layer", "osm_id", "fclass", "geometry"]
)

print(f"Exact duplicate records: {exact_duplicates.sum():,}")

osm_amenities = osm_amenities.loc[~exact_duplicates].copy()

# Repeated names and categories are candidates for review,
# not confirmed duplicates. Different branches can share a name.
named = osm_amenities.dropna(subset=["name"]).copy()
named["name_normalized"] = named["name"].str.casefold()

potential_duplicates = named[
    named.duplicated(
        subset=["name_normalized", "fclass"],
        keep=False
    )
].sort_values(["name_normalized", "fclass"])

print(f"Records with repeated name/category: {len(potential_duplicates):,}")

display(
    potential_duplicates[
        ["name", "fclass", "source_layer", "osm_id"]
    ].head(20)
)

Exact duplicate records: 0
Records with repeated name/category: 11,863


,name,fclass,source_layer,osm_id
71320,#1,pitch,gis_osm_pois_a_free,511370125
71513,#1,pitch,gis_osm_pois_a_free,512180306
71521,#1,pitch,gis_osm_pois_a_free,512184008
71321,#2,pitch,gis_osm_pois_a_free,511370126
71514,#2,pitch,gis_osm_pois_a_free,512180307
71522,#2,pitch,gis_osm_pois_a_free,512184009
71322,#3,pitch,gis_osm_pois_a_free,511370129
71515,#3,pitch,gis_osm_pois_a_free,512180308
71520,#3,pitch,gis_osm_pois_a_free,512184007
71323,#4,pitch,gis_osm_pois_a_free,511370131


In [15]:
# Separate named point and polygon records
poi_points = named[
    named.geometry.geom_type.eq("Point")
].copy()

poi_areas = named[
    named.geometry.geom_type.isin(["Polygon", "MultiPolygon"])
].copy()

# Find points inside or on the boundary of amenity polygons
matches = gpd.sjoin(
    poi_points[
        ["record_id", "name", "name_normalized", "fclass", "geometry"]
    ],
    poi_areas[
        ["record_id", "name_normalized", "fclass", "geometry"]
    ],
    how="inner",
    predicate="intersects",
    lsuffix="point",
    rsuffix="area"
)

# Keep only matching names AND categories
duplicate_candidates = matches[
    matches["name_normalized_point"].eq(
        matches["name_normalized_area"]
    )
    & matches["fclass_point"].eq(matches["fclass_area"])
].copy()

print(f"Potential point/polygon duplicate pairs: {len(duplicate_candidates):,}")

display(
    duplicate_candidates[
        ["name", "fclass_point", "record_id_point", "record_id_area"]
    ].head(20)
)

Potential point/polygon duplicate pairs: 103


,name,fclass_point,record_id_point,record_id_area
69,CVS/pharmacy,pharmacy,gis_osm_pois_free:321247238,gis_osm_pois_a_free:239298946
95,Armstrong Park,park,gis_osm_pois_free:357767398,gis_osm_pois_a_free:33062283
272,Glenburnie Park,park,gis_osm_pois_free:357773562,gis_osm_pois_a_free:468956084
758,Watts Park,park,gis_osm_pois_free:357796817,gis_osm_pois_a_free:569696991
933,Singletary Lake State Park,park,gis_osm_pois_free:357797802,gis_osm_pois_a_free:190496
963,Jones Lake State Park,park,gis_osm_pois_free:357798604,gis_osm_pois_a_free:190502
1014,Mazarick Park,park,gis_osm_pois_free:357799059,gis_osm_pois_a_free:32751942
1240,Abernathy Ball Field,park,gis_osm_pois_free:357800117,gis_osm_pois_a_free:569697399
1707,Lake Daniel Park,park,gis_osm_pois_free:357810422,gis_osm_pois_a_free:486669934
1839,Washington Terrace Park,park,gis_osm_pois_free:357813158,gis_osm_pois_a_free:910704789


In [16]:
# Count how many different features match each point and polygon
pairs = duplicate_candidates[
    ["record_id_point", "record_id_area"]
].drop_duplicates()

point_matches = pairs.groupby("record_id_point")[
    "record_id_area"
].nunique()

area_matches = pairs.groupby("record_id_area")[
    "record_id_point"
].nunique()

# Only consolidate unambiguous one-to-one matches
clear_pairs = pairs[
    pairs["record_id_point"].map(point_matches).eq(1)
    & pairs["record_id_area"].map(area_matches).eq(1)
].copy()

# Keep an audit table of the point records excluded
osm_duplicate_audit = clear_pairs.copy()
osm_duplicate_audit["reason"] = (
    "Same name and category; point intersects polygon; one-to-one match"
)

# Create a separate cleaned table, retaining the polygon representation
osm_amenities_clean = osm_amenities[
    ~osm_amenities["record_id"].isin(clear_pairs["record_id_point"])
].copy()

print(f"Point records excluded: {len(clear_pairs):,}")
print(f"Remaining amenity records: {len(osm_amenities_clean):,}")
print(f"Ambiguous pairs retained: {len(pairs) - len(clear_pairs):,}")

display(
    osm_amenities_clean.groupby("amenity_group")
    .size()
    .reset_index(name="records")
)

Point records excluded: 103
Remaining amenity records: 60,273
Ambiguous pairs retained: 0


,amenity_group,records
0,entertainment_and_culture,1381
1,food_and_nightlife,15370
2,outdoor_recreation,10071
3,shopping_and_daily_needs,5795
4,sports_and_fitness,27656


In [17]:
OUTPUT_DIR = DATA_DIR / "processed"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

osm_amenities_clean.to_file(
    OUTPUT_DIR / "osm_amenities_clean.gpkg",
    layer="amenities",
    driver="GPKG",
    engine="pyogrio"
)

osm_duplicate_audit.to_csv(
    OUTPUT_DIR / "osm_duplicate_audit.csv",
    index=False
)

print("Saved cleaned OSM amenities and duplicate audit.")

Saved cleaned OSM amenities and duplicate audit.


In [19]:
BEACH_URL = (
    "https://services2.arcgis.com/kCu40SDxsCGcuUWO/"
    "ArcGIS/rest/services/DCM_Beach_and_Waterfront_Access/"
    "FeatureServer/0"
)

def arcgis_request(url, params):
    response = requests.post(url, data=params, timeout=120)
    response.raise_for_status()
    data = response.json()

    if "error" in data:
        raise RuntimeError(data["error"])

    return data


# Get IDs first so we can request every record in manageable batches
id_response = arcgis_request(
    f"{BEACH_URL}/query",
    {
        "where": "1=1",
        "returnIdsOnly": "true",
        "f": "json"
    }
)

object_ids = sorted(id_response.get("objectIds") or [])

if not object_ids:
    raise ValueError("The beach API returned no record IDs.")

features = []

for start in range(0, len(object_ids), 500):
    batch_ids = object_ids[start:start + 500]

    batch = arcgis_request(
        f"{BEACH_URL}/query",
        {
            "objectIds": ",".join(map(str, batch_ids)),
            "outFields": "*",
            "returnGeometry": "true",
            "outSR": 4326,
            "f": "geojson"
        }
    )

    if batch.get("exceededTransferLimit"):
        raise RuntimeError("API response was truncated. Reduce batch size.")

    features.extend(batch["features"])

beach_access = gpd.GeoDataFrame.from_features(
    features,
    crs="EPSG:4326"
)

# Verify IDs as well as the number of records
id_field = id_response["objectIdFieldName"]
received_ids = set(beach_access[id_field])

if (
    received_ids != set(object_ids)
    or len(beach_access) != len(object_ids)
):
    raise ValueError("Retrieved records do not match the requested IDs.")

print(f"Retrieved {len(beach_access):,} coastal-access records.")
print("Columns:", beach_access.columns.tolist())

display(beach_access.head(3))

Retrieved 911 coastal-access records.
Columns: ['geometry', 'ACCESS_ID', 'OBJECTID', 'PLACE', 'ACCESS_LOC', 'LOCL_SPONS', 'CAMA_SIGN', 'PARKING', 'DUNE_WALK', 'OVERLOOK', 'HAND_DUNE', 'HAND_OVRLK', 'HAND_BEACH', 'HAND_PRKG', 'RESTRM', 'SHOWER', 'PIC_TABLE', 'VEH_ACCESS', 'BIKE_RACK', 'SURV_DATE', 'COMMENT_', 'PRKG_COMMENTS', 'COUNTY', 'DCM_RO', 'NCDCM_ID', 'PHOTO', 'PRK_FEE', 'DCM_Funded', 'ACESS_TYPE', 'KayakCanoeLaunch', 'PARKING_OPTIONS', 'BeachWheelchair', 'BWC_Info', 'PaidParkURL', 'WaterAccessTYPE', 'BeachMat', 'Land_Status', 'ACCESSIBILITY_NOTES', 'LIFEGUARDS']


,geometry,ACCESS_ID,OBJECTID,PLACE,ACCESS_LOC,LOCL_SPONS,CAMA_SIGN,PARKING,DUNE_WALK,OVERLOOK,...,KayakCanoeLaunch,PARKING_OPTIONS,BeachWheelchair,BWC_Info,PaidParkURL,WaterAccessTYPE,BeachMat,Land_Status,ACCESSIBILITY_NOTES,LIFEGUARDS
0,POINT (-77.64421 34.35279),Ocean Blvd (O-16),1,Topsail Beach,Ocean and Florida,Topsail Beach,Yes,0.0,Y,No,...,No,On-Street Parking,Y,https://topsailbeachnc.gov/Departments/Police/...,n/a,Ocean,"Yes, Summer Only",Municipality,NaN,NaN
1,POINT (-77.52441 34.43761),Beach Access #8 (Jones Ave),2,Surf City,1300 N Shore Dr,Surf City,Yes,4.0,Y,No,...,No,Onsite Parking,Y,https://www.surfcitync.gov/2413/Beach-Wheelchairs,https://www.surfcitync.gov/2425/Beach-Parking,Ocean,NaN,Municipality,NaN,NaN
2,POINT (-77.46961 34.4686),OC BA-23,3,North Topsail Beach,7th Ave & Topsail Rd,Onslow County,Yes,0.0,Y,No,...,No,No Parking,Y,https://www.ntbnc.org/_files/ugd/e4899e_ed33ab...,n/a,Ocean,NaN,Municipality,NaN,NaN


In [20]:
display(
    beach_access["ACESS_TYPE"]
    .value_counts(dropna=False)
    .rename_axis("access_type")
    .reset_index(name="records")
)

print(
    "Missing or empty geometries:",
    (
        beach_access.geometry.isna()
        | beach_access.geometry.is_empty
    ).sum()
)

,access_type,records
0,Beach,636
1,Shoreline,266
2,NaN,9


Missing or empty geometries: 0


In [21]:
beach_access_clean = beach_access.copy()

beach_access_clean["access_type_clean"] = (
    beach_access_clean["ACESS_TYPE"]
    .astype("string")
    .str.strip()
    .replace("", pd.NA)
    .fillna("Unknown")
)

# Remove unusable locations
usable_geometry = (
    beach_access_clean.geometry.notna()
    & ~beach_access_clean.geometry.is_empty
)

print(f"Records without usable geometry: {(~usable_geometry).sum():,}")

beach_access_clean = (
    beach_access_clean.loc[usable_geometry]
    .copy()
    .to_crs("EPSG:32119")
)

# Separate subsets for later distance calculations
beach_points = beach_access_clean[
    beach_access_clean["access_type_clean"].eq("Beach")
].copy()

shoreline_points = beach_access_clean[
    beach_access_clean["access_type_clean"].eq("Shoreline")
].copy()

beach_access_clean.to_file(
    OUTPUT_DIR / "nc_coastal_access_clean.gpkg",
    layer="coastal_access",
    driver="GPKG",
    engine="pyogrio"
)

print("Saved coastal-access data.")

Records without usable geometry: 0
Saved coastal-access data.


In [22]:
RAIL_URL = (
    "https://gis11.services.ncdot.gov/arcgis/rest/services/"
    "NCDOT_RailSystem/MapServer/0"
)

# Get all facility IDs
rail_id_response = arcgis_request(
    f"{RAIL_URL}/query",
    {
        "where": "1=1",
        "returnIdsOnly": "true",
        "f": "json"
    }
)

rail_ids = sorted(rail_id_response.get("objectIds") or [])

if not rail_ids:
    raise ValueError("The rail API returned no record IDs.")

rail_features = []

for start in range(0, len(rail_ids), 100):
    batch_ids = rail_ids[start:start + 100]

    batch = arcgis_request(
        f"{RAIL_URL}/query",
        {
            "objectIds": ",".join(map(str, batch_ids)),
            "outFields": "*",
            "returnGeometry": "true",
            "outSR": 4326,
            "f": "geojson"
        }
    )

    if batch.get("exceededTransferLimit"):
        raise RuntimeError("Rail response was truncated.")

    rail_features.extend(batch["features"])

rail_facilities = gpd.GeoDataFrame.from_features(
    rail_features,
    crs="EPSG:4326"
)

# Confirm that every requested record was retrieved exactly once
rail_id_field = rail_id_response["objectIdFieldName"]

if (
    set(rail_facilities[rail_id_field]) != set(rail_ids)
    or len(rail_facilities) != len(rail_ids)
):
    raise ValueError("Retrieved rail records do not match requested IDs.")

print(f"Retrieved {len(rail_facilities):,} rail facilities.")

display(
    rail_facilities["FacType"]
    .value_counts(dropna=False)
    .rename_axis("facility_type")
    .reset_index(name="records")
)

Retrieved 147 rail facilities.


,facility_type,records
0,Transload,93
1,Yard,23
2,Station,16
3,Station| Amtrak Bus,11
4,Intermodal,4


In [24]:
# Select station candidates from the downloaded rail facilities
station_mask = (
    rail_facilities["FacType"]
    .astype("string")
    .str.contains("Station", case=False, na=False)
)

rail_station_candidates = rail_facilities.loc[station_mask].copy()

display(
    rail_station_candidates[
        ["LocationNa", "FacType", "Railroad", "County"]
    ].sort_values(["FacType", "LocationNa"])
)

,LocationNa,FacType,Railroad,County
85,Burlington,Station,ATK,Alamance
86,Cary,Station,ATK,Wake
87,Charlotte,Station,ATK,Mecklenburg
88,Durham,Station,ATK,Durham
89,Fayetteville,Station,ATK,Cumberland
99,Gastonia,Station,ATK,Gaston
90,Greensboro,Station,ATK,Guilford
91,Hamlet,Station,ATK,Richmond
92,High Point,Station,ATK,Guilford
93,Kannapolis,Station,ATK,Cabarrus


In [25]:
facility_type = (
    rail_station_candidates["FacType"]
    .astype("string")
    .str.strip()
)

rail_stations = rail_station_candidates[
    facility_type.eq("Station")
].copy()

amtrak_bus_stops = rail_station_candidates[
    facility_type.eq("Station| Amtrak Bus")
].copy()

rail_stations["access_mode"] = "passenger_rail"
amtrak_bus_stops["access_mode"] = "amtrak_bus"

transport_access = gpd.GeoDataFrame(
    pd.concat(
        [rail_stations, amtrak_bus_stops],
        ignore_index=True
    ),
    geometry="geometry",
    crs=rail_facilities.crs
)

# Check locations before saving
usable = (
    transport_access.geometry.notna()
    & ~transport_access.geometry.is_empty
)

print(f"Records without usable geometry: {(~usable).sum():,}")

transport_access = (
    transport_access.loc[usable]
    .copy()
    .to_crs("EPSG:32119")
)

transport_access.to_file(
    OUTPUT_DIR / "nc_rail_and_bus_access.gpkg",
    layer="transport_access",
    driver="GPKG",
    engine="pyogrio"
)

display(
    transport_access.groupby("access_mode")
    .size()
    .reset_index(name="records")
)

Records without usable geometry: 0


,access_mode,records
0,amtrak_bus,11
1,passenger_rail,16


In [26]:
osm_transport_points = gpd.read_file(
    OSM_PATH,
    layer="gis_osm_transport_free",
    engine="pyogrio"
)

osm_transport_areas = gpd.read_file(
    OSM_PATH,
    layer="gis_osm_transport_a_free",
    engine="pyogrio"
)

transport_categories = pd.concat(
    [
        osm_transport_points["fclass"]
        .value_counts().rename("points"),

        osm_transport_areas["fclass"]
        .value_counts().rename("areas")
    ],
    axis=1
).fillna(0).astype(int)

print(transport_categories.to_string())

                 points  areas
fclass                        
bus_stop           5388      7
airport             327    141
ferry_terminal       42      4
railway_station      41      8
tram_stop            34      0
helipad              32    235
taxi                  8      1
bus_station           7     28
airfield              4     10
railway_halt          3      0
apron                 0    247


In [27]:
for label, df in [
    ("Points", osm_transport_points),
    ("Areas", osm_transport_areas)
]:
    airport_mask = df["fclass"].str.contains(
        "airport|airfield|aerodrome|terminal",
        case=False,
        na=False
    )

    print(f"\n{label}: airport and terminal candidates")

    display(
        df.loc[airport_mask, ["osm_id", "name", "fclass"]]
        .head(30)
    )


Points: airport and terminal candidates


,osm_id,name,fclass
0,169326249,Bayview-Aurora Ferry,ferry_terminal
1,169375507,Sans Souci Ferry,ferry_terminal
2,169832965,Cedar Island Ferry,ferry_terminal
10,170289124,,ferry_terminal
11,170289834,,ferry_terminal
12,170321159,Hatteras Ferry Terminal,ferry_terminal
15,172442228,,ferry_terminal
18,172678275,,ferry_terminal
21,367908097,Donald's Air Park,airport
22,367908114,Twin Oak Airport,airport



Areas: airport and terminal candidates


,osm_id,name,fclass
2,31359838,Smith-Reynolds International Airport,airport
9,38108891,Fayetteville Regional Airport,airport
16,58898864,Plymouth Municipal Airport,airport
21,89309763,Lake Ridge Aero Park,airport
25,103266897,Albert J Ellis Airport,airport
53,118715920,Odell Williamson Municipal Airport,airport
55,128227285,Ocracoke Island Airport,airport
65,209063360,Raleigh-Durham International Airport,airport
67,222784159,Pope Army Airfield,airport
68,222784159,Pope Army Airfield,airfield


In [28]:
airport_parts = []

for source_layer, df in [
    ("gis_osm_transport_free", osm_transport_points),
    ("gis_osm_transport_a_free", osm_transport_areas)
]:
    subset = df[
        df["fclass"].isin(["airport", "airfield"])
    ].copy()

    subset["source_layer"] = source_layer
    airport_parts.append(subset.to_crs("EPSG:32119"))

airport_inventory = gpd.GeoDataFrame(
    pd.concat(airport_parts, ignore_index=True),
    geometry="geometry",
    crs="EPSG:32119"
)

# Compact list of distinct names for review
airport_names = (
    airport_inventory["name"]
    .astype("string")
    .str.strip()
    .replace("", pd.NA)
    .dropna()
    .drop_duplicates()
    .sort_values()
)

print(f"Airport/airfield records: {len(airport_inventory):,}")
print("\nDistinct airport names:")
print(airport_names.to_string(index=False))

Airport/airfield records: 482

Distinct airport names:
                                     Adams Airport
                           Aero Plantation Airport
                                Air Harbor Airport
         Alamance Regional Medical Center Heliport
                            Albert J Ellis Airport
                          Alexander County Airport
                                     Allison Woods
                                       Als Airport
                     Angel Medical Center Heliport
           Anson County Airport (Jeff Cloud Field)
                                     Arant Airport
                               Ashe County Airport
                         Asheboro Regional Airport
                        Asheville Regional Airport
                   Atrium Health Cabarrus Heliport
                 Atrium Health Harrisburg Heliport
             Atrium Health Kings Mountain Heliport
                     Atrium Health Stanly Heliport
                      Atriu

In [29]:
# Name patterns for NC's commercial passenger airport shortlist
airport_patterns = {
    "Charlotte Douglas": r"charlotte.*douglas",
    "Raleigh-Durham": r"raleigh.*durham",
    "Piedmont Triad": r"piedmont.*triad",
    "Asheville": r"asheville.*regional",
    "Wilmington": r"wilmington.*international",
    "Fayetteville": r"fayetteville.*regional",
    "Albert J. Ellis": r"albert.*ellis",
    "Coastal Carolina": r"coastal.*carolina",
    "Pitt-Greenville": r"pitt.*greenville",
    "Concord-Padgett": r"concord.*(padgett|regional)"
}

airport_matches = []

for airport, pattern in airport_patterns.items():
    matches = airport_inventory[
        airport_inventory["name"]
        .astype("string")
        .str.contains(pattern, case=False, regex=True, na=False)
    ].copy()

    matches["airport_group"] = airport
    airport_matches.append(matches)

commercial_airport_candidates = gpd.GeoDataFrame(
    pd.concat(airport_matches, ignore_index=True),
    geometry="geometry",
    crs=airport_inventory.crs
)

# Small summary that won't flood the output
summary = pd.DataFrame({
    "airport": list(airport_patterns),
    "matching_osm_records": [
        commercial_airport_candidates["airport_group"].eq(name).sum()
        for name in airport_patterns
    ]
})

display(summary)

C:\Users\kyleh\AppData\Local\Temp\ipykernel_27580\955052293.py:21: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  .str.contains(pattern, case=False, regex=True, na=False)


,airport,matching_osm_records
0,Charlotte Douglas,1
1,Raleigh-Durham,1
2,Piedmont Triad,0
3,Asheville,1
4,Wilmington,1
5,Fayetteville,1
6,Albert J. Ellis,1
7,Coastal Carolina,1
8,Pitt-Greenville,1
9,Concord-Padgett,1


In [30]:
# Search original transportation layers, including other feature classes
for label, df in [
    ("Points", osm_transport_points),
    ("Areas", osm_transport_areas)
]:
    mask = (
        df["name"]
        .astype("string")
        .str.contains(
            r"piedmont|triad|greensboro|\bGSO\b",
            case=False,
            regex=True,
            na=False
        )
    )

    print(f"\n{label}: possible Piedmont Triad matches")

    print(
        df.loc[mask, ["osm_id", "name", "fclass"]]
        .to_string(index=False)
    )


Points: possible Piedmont Triad matches
     osm_id                                         name          fclass
  369029187                 Southeast Greensboro Airport         airport
  672576759        S Greensboro St at Municipal Park Lot        bus_stop
  672576841      South Greensboro Street at Trailer Park        bus_stop
 1042008276                     Greensboro North Airport         airport
 2574007085     North Greensboro Street at Harris Teeter        bus_stop
 5856773225   North Greensboro Street at Williams Street        bus_stop
 5856773226 North Greensboro Street at Morningside Drive        bus_stop
 5856773242        North Greensboro Street at Oak Avenue        bus_stop
 5856773243      North Greensboro Street at Cheek Street        bus_stop
 5856773244 North Greensboro Street at Weathervane Drive        bus_stop
 5856773245      North Greensboro Street at Milton Drive        bus_stop
 5856773263       North Greensboro Street at Todd Street        bus_stop
 585677326

### Data prepared so far

- **OpenStreetMap amenities:** We grouped selected features into food and nightlife, shopping and daily needs, sports and fitness, entertainment and culture, and outdoor recreation. We excluded 103 likely duplicate point representations where a point intersected a polygon with the same name and category in a one-to-one match. The cleaned dataset contains 60,273 records and has been saved with a duplicate audit table.
- **Protected lands:** We filtered PAD-US to 2,830 open-access land records and assigned broader recreation and land categories. These records may include overlapping boundaries or separate pieces of the same named area.
- **Coastal access:** We retrieved 911 records through the NC coastal-access API: 636 labeled Beach, 266 Shoreline, and 9 with an unknown access type. We prepared and saved the cleaned data, keeping these classifications separate.
- **Rail and bus access:** We retrieved 147 rail facilities and separated 16 passenger-station records from 11 Amtrak bus locations. All 27 had usable geometry, and the combined access dataset has been saved.
- **Airports:** We identified OSM candidates for nine airports on our commercial passenger airport shortlist. Piedmont Triad has not yet been matched and can be added later from a documented source.

### What these records represent

The datasets describe mapped locations and land features. Counts do not necessarily represent unique venues, and OSM facilities are not automatically open to the public. Our duplicate check addressed one specific type of repeated representation; other duplicates may remain.

Airport coverage is incomplete, and station classifications reflect the downloaded source rather than a check of current service schedules. Airport boundaries also do not identify passenger terminal entrances.

### Next steps

Before closing this preparation stage, we need to save the PAD-US and airport candidate outputs. We will then align the housing geography and calculate features such as distance to beach access, passenger stations, airports, and nearby amenities.

No accessibility distances, travel times, or final recommendations have been calculated yet. Initial distance measures will be labeled as straight-line distances, with driving times treated as a separate future step.

In [31]:
padus_open.to_file(
    OUTPUT_DIR / "padus_open_clean.gpkg",
    layer="open_access_land",
    driver="GPKG",
    engine="pyogrio"
)

print(f"Saved {len(padus_open):,} open-access land records.")

Saved 2,830 open-access land records.


In [32]:
commercial_airport_candidates.to_file(
    OUTPUT_DIR / "nc_airport_candidates.gpkg",
    layer="airport_candidates",
    driver="GPKG",
    engine="pyogrio"
)

print(f"Saved {len(commercial_airport_candidates):,} airport candidate records.")

Saved 9 airport candidate records.
